# LeetCode 402 — Remove K Digits bằng Python

Link đề: https://leetcode.com/problems/remove-k-digits/ · Độ khó: **Medium** · Chủ đề: Stack, Greedy, String, Monotonic Stack

Skeleton LeetCode:

```python
class Solution:
    def removeKdigits(self, num: str, k: int) -> str:
        ...
```

Ghi chú: phần phát biểu đề được dịch sang tiếng Việt từ đề gốc; mọi ví dụ và đáp án trong notebook đều được chạy lại và đối chiếu với một oracle vét cạn độc lập (thử mọi cách xóa `k` chữ số trên input nhỏ).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho một chuỗi số `num` và số nguyên `k`, **xóa đúng `k` chữ số** (giữ nguyên thứ tự các chữ số còn lại) sao cho số tạo thành **nhỏ nhất có thể**; kết quả không có số `0` thừa ở đầu, và nếu rỗng thì trả về `"0"`.
2. Nắm công cụ Python: so sánh chuỗi, `list` dùng như **stack** (`append`, `pop`, `[-1]`), cắt lát `s[:m]`, `str.lstrip('0')`, `or '0'`, `''.join`, `itertools.combinations`, `bisect`.
3. Đi từ cách **vét cạn mọi tổ hợp** → **xóa tham lam từng lần `O(n*k)`** → **monotonic stack `O(n)`** → **chọn từng chữ số kết quả bằng danh sách vị trí + `bisect`**.
4. Hiểu mấu chốt: với hai số cùng độ dài, so sánh **từ trái sang phải**; nếu một chữ số lớn hơn chữ số ngay sau nó thì **xóa chữ số lớn đó** luôn có lợi. Đó là lý do ta dùng stack "không giảm".
5. Tránh các lỗi kinh điển: quên bỏ số `0` ở đầu, dùng `>=` thay vì `>` khi pop, cắt `stk[:-k]` khi `k = 0`, đổi sang `int` với chuỗi quá dài, quên trường hợp kết quả rỗng.

## 2. Đề bài & ví dụ

Cho chuỗi `num` biểu diễn một số nguyên không âm và một số nguyên `k`. Hãy trả về **số nhỏ nhất có thể** (dưới dạng chuỗi) sau khi **xóa đúng `k` chữ số** khỏi `num`.

**Ví dụ 1:** `num = "1432219"`, `k = 3` → `"1219"`. Xóa các chữ số `4`, `3`, `2` để được `1219`, nhỏ nhất.

**Ví dụ 2:** `num = "10200"`, `k = 1` → `"200"`. Xóa chữ số `1` ở đầu, còn `0200`; kết quả **không được có số `0` thừa ở đầu** nên là `"200"`.

**Ví dụ 3:** `num = "10"`, `k = 2` → `"0"`. Xóa hết các chữ số thì còn "không có gì", ta quy ước đó là `0`.

**Ràng buộc:**

- `1 <= k <= num.length <= 10^5`
- `num` chỉ gồm các chữ số.
- `num` không có số `0` thừa ở đầu (trừ chính số `0`).

In [1]:
from itertools import combinations
from bisect import bisect_left
import random
import time
import sys
from typing import List

examples = [
    ("1432219", 3, "1219"),
    ("10200", 1, "200"),
    ("10", 2, "0"),
]
for num, k, expected in examples:
    print(f"num={num!r}, k={k} -> kỳ vọng {expected!r}")

num='1432219', k=3 -> kỳ vọng '1219'
num='10200', k=1 -> kỳ vọng '200'
num='10', k=2 -> kỳ vọng '0'


### 2.1. Đọc kỹ đề

- Ta **xóa** chữ số, không được **đổi chỗ** các chữ số còn lại. Kết quả là một *dãy con* (subsequence) của `num` có độ dài `len(num) - k`.
- "Nhỏ nhất" theo **giá trị số**. Hai số cùng độ dài thì số nào có chữ số khác biệt đầu tiên (từ trái) nhỏ hơn là số nhỏ hơn. Vì vậy chữ số **bên trái quan trọng hơn**.
- Kết quả **không có `0` ở đầu**: `"0200"` phải viết thành `"200"`. Nếu mọi chữ số bị xóa hoặc toàn `0` thì trả về `"0"`.
- `num` dài tới `10^5` chữ số: **không thể** dùng `int(num)` (Python giới hạn mặc định 4300 chữ số khi đổi chuỗi sang `int`) và không thể vét cạn mọi tổ hợp.

## 3. Kiến thức nền

### 3.1. So sánh chuỗi chữ số

Python so sánh hai chuỗi **từng ký tự từ trái sang phải** (thứ tự từ điển). Với chuỗi chỉ gồm chữ số, `'0' < '1' < ... < '9'`. Khi hai chuỗi **cùng độ dài**, so sánh từ điển cho cùng kết quả với so sánh giá trị số. Nhưng khác độ dài thì **khác**: `"9" > "10"` theo chuỗi nhưng `9 < 10` theo số.

In [2]:
print('3' < '5', '9' > '10', 9 > 10)             # True True False  (khác độ dài: chuỗi != số)
print("0200" < "0300", int("0200") < int("0300"))  # cùng độ dài: hai cách so sánh khớp nhau
print("1219" < "1319", min("1432", "1243", "1423"))

True True False
True True
True 1243


### 3.2. `list` dùng như một stack (ngăn xếp)

Stack là cấu trúc "vào sau, ra trước". Với `list` của Python:

- `stk.append(x)`: đẩy `x` lên đỉnh.
- `stk[-1]`: xem phần tử trên đỉnh (không lấy ra).
- `stk.pop()`: lấy phần tử trên đỉnh ra và trả về nó.
- `if stk:` / `while stk:`: kiểm tra stack còn phần tử hay không (rỗng là `False`).

In [3]:
stk = []
for c in "3142":
    stk.append(c)
print(stk, "đỉnh =", stk[-1])
print("pop ->", stk.pop(), stk)
print("rỗng?", not [], "| còn phần tử?", bool(stk))

['3', '1', '4', '2'] đỉnh = 2
pop -> 2 ['3', '1', '4']
rỗng? True | còn phần tử? True


### 3.3. Cắt lát, `lstrip` và `or`

- `s[:m]` lấy `m` ký tự đầu; `s[:-j]` bỏ `j` ký tự cuối (cẩn thận khi `j = 0`: `s[:-0]` là **chuỗi rỗng**, vì `-0 == 0`).
- `s.lstrip('0')` xóa mọi ký tự `'0'` ở **đầu** chuỗi.
- `a or b` trả về `a` nếu `a` "thật" (chuỗi không rỗng), ngược lại trả về `b`. Do đó `"".lstrip('0') or '0'` cho `'0'`.

In [4]:
s = "0012300"
print(s[:3], s[:-2], repr(s[:-0]))              # '001' '00123' ''   <- s[:-0] là rỗng!
print(s.lstrip('0'), repr("000".lstrip('0')), repr("000".lstrip('0') or '0'))
print("".join(['1', '2', '3']), "".join([]) == "")

001 00123 ''
12300 '' '0'
123 True


### 3.4. `itertools.combinations` và `bisect`

`combinations(range(5), 3)` sinh mọi cách chọn 3 chỉ số tăng dần từ `0..4` (đúng 10 cách). Ta sẽ dùng nó để chọn **các chữ số giữ lại** trong cách vét cạn.

`bisect_left(a, x)` trên danh sách **đã sắp xếp** `a` cho vị trí đầu tiên mà `a[pos] >= x` — dùng để tìm "lần xuất hiện đầu tiên của một chữ số tại vị trí `>= start`".

In [5]:
print(list(combinations(range(4), 2)))
pos0 = [1, 3, 4]                                 # các vị trí của chữ số '0' trong một chuỗi nào đó
print(bisect_left(pos0, 0), bisect_left(pos0, 2), bisect_left(pos0, 5))   # 0 1 3 (3 = hết danh sách)

[(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
0 1 3


### 3.5. Số cần giữ lại

Nếu `num` có `n` chữ số và xóa đúng `k`, kết quả có đúng `m = n - k` chữ số (trước khi bỏ `0` đầu). Vì xóa **đúng `k`** (không ít hơn), ta luôn xóa đủ: nếu còn dư lượt xóa sau khi duyệt, xóa ở **cuối**.

## 4. Ý tưởng nền tảng: tính tay

Xét `num = "1432219"`, `k = 3`. Ta muốn số nhỏ nhất, nên muốn các chữ số **bên trái** càng nhỏ càng tốt.

- Nhìn cặp đầu `1, 4`: `1 < 4`, tăng dần, chưa có lý do xóa.
- Cặp `4, 3`: `4 > 3`. Nếu xóa `4`, chữ số ở vị trí thứ hai trở thành `3` (nhỏ hơn `4`) ⇒ số nhỏ đi. Xóa `4` ⇒ `132219`, còn `k = 2`.
- Cặp `3, 2`: `3 > 2`. Xóa `3` ⇒ `12219`, còn `k = 1`.
- Cặp `2, 2`: bằng nhau, xóa cái nào cũng như nhau, chưa có lợi. Cặp `2, 1`: `2 > 1`, xóa `2` ⇒ `1219`, còn `k = 0` ✓.

**Quy tắc:** nếu một chữ số **lớn hơn chữ số ngay sau nó**, xóa chữ số đó sẽ làm số nhỏ đi (vì vị trí của nó được thay bằng một chữ số nhỏ hơn, các vị trí bên trái không đổi). Nếu **không có** cặp giảm nào (dãy không giảm, ví dụ `12345`), xóa chữ số **cuối cùng** là tốt nhất vì nó ở vị trí ít quan trọng nhất.

## 5. Cách 1 — Brute Force: thử mọi cách giữ lại `n - k` chữ số

**Ý tưởng:** chọn mọi tập `n - k` chỉ số (tăng dần) để **giữ lại**, ghép thành chuỗi, lấy chuỗi nhỏ nhất. Mọi chuỗi ứng viên có **cùng độ dài** `n - k` nên so sánh từ điển = so sánh số. Cuối cùng bỏ `0` ở đầu.

In [6]:
def remove_k_brute(num, k):
    n = len(num)
    m = n - k
    best = None
    for keep in combinations(range(n), m):          # mỗi tổ hợp = các chỉ số được giữ lại
        cand = "".join(num[i] for i in keep)
        if best is None or cand < best:
            best = cand
    if best is None:                                # m = 0: không giữ gì
        best = ""
    return best.lstrip("0") or "0"


for num, k, expected in examples:
    got = remove_k_brute(num, k)
    print(f"{num!r}, k={k} -> {got!r} | kỳ vọng {expected!r}")
    assert got == expected

'1432219', k=3 -> '1219' | kỳ vọng '1219'
'10200', k=1 -> '200' | kỳ vọng '200'
'10', k=2 -> '0' | kỳ vọng '0'


**Độ phức tạp:** có `C(n, k)` tổ hợp, mỗi tổ hợp dựng chuỗi `O(n)` ⇒ Time `O(C(n,k) * n)`, Space `O(n)`.

**Nút thắt:** `C(n, k)` bùng nổ rất nhanh. Với `n = 30`, `k = 15` đã là hơn 155 triệu tổ hợp. Hãy đếm thử để thấy.

In [7]:
from math import comb
for n, k in [(10, 3), (20, 10), (30, 15), (100, 50)]:
    print(f"n={n:>3}, k={k:>2}: C(n,k) = {comb(n, k):,}")
print("với n = 10^5 thì con số có hàng chục nghìn chữ số — vét cạn bất khả thi")

n= 10, k= 3: C(n,k) = 120
n= 20, k=10: C(n,k) = 184,756
n= 30, k=15: C(n,k) = 155,117,520
n=100, k=50: C(n,k) = 100,891,344,545,564,193,334,812,497,256
với n = 10^5 thì con số có hàng chục nghìn chữ số — vét cạn bất khả thi


## 6. Ý tưởng thuật toán / bước đột phá

Từ mục 4, ta rút ra quy tắc tham lam:

> Mỗi lần xóa một chữ số, hãy xóa chữ số **đầu tiên** (từ trái) mà **lớn hơn chữ số ngay sau nó**. Nếu không có chữ số nào như vậy thì xóa chữ số **cuối cùng**.

**Vì sao xóa chữ số đầu tiên như vậy là tốt nhất?** Gọi vị trí đó là `i`, tức `num[i] > num[i+1]` và mọi cặp trước đó không giảm: `num[0] <= num[1] <= ... <= num[i]`. Xóa `num[i]` đưa `num[i+1]` (nhỏ hơn) lên vị trí `i`, các vị trí `0..i-1` giữ nguyên ⇒ số nhỏ đi **tại vị trí `i`**. Xóa bất kỳ chữ số nào ở vị trí `j < i` thì vị trí `j` nhận `num[j+1] >= num[j]` — **không tốt hơn**. Xóa ở vị trí `j > i` thì vị trí `i` vẫn là `num[i]` — **không tốt hơn** việc đưa `num[i+1]` lên.

Làm như vậy `k` lần là xong; đó là **Cách 2**. Nhưng mỗi lần quét lại từ đầu tốn `O(n)` ⇒ tổng `O(n*k)`. **Bước đột phá:** các lần quét lặp lại cùng một phần đầu chuỗi. Ta dùng một **stack** để nhớ "phần đầu đã không giảm", và khi gặp chữ số mới nhỏ hơn đỉnh stack thì pop (xóa) đỉnh — chỉ quét **một lần** ⇒ `O(n)`.

## 7. Các cách tiếp theo

### Cách 2 — Xóa tham lam từng lần (chữ số đầu tiên lớn hơn chữ số kế tiếp)

**Giải thích từng đoạn:**

1. Lặp `k` lần.
2. Mỗi lần quét `i` từ `0` đến `len(s) - 2`; nếu `s[i] > s[i+1]` thì xóa `s[i]` (`s = s[:i] + s[i+1:]`) và dừng quét.
3. Nếu quét hết mà không gặp cặp giảm nào, xóa ký tự cuối: `s = s[:-1]`.
4. Cuối cùng `lstrip('0') or '0'`.

In [8]:
def remove_k_greedy_repeat(num, k):
    s = num
    for _ in range(k):
        removed = False
        for i in range(len(s) - 1):
            if s[i] > s[i + 1]:                      # chữ số đầu tiên lớn hơn chữ số kế tiếp
                s = s[:i] + s[i + 1:]
                removed = True
                break
        if not removed:                              # dãy không giảm: xóa chữ số cuối
            s = s[:-1]
    return s.lstrip("0") or "0"


for num, k, expected in examples:
    got = remove_k_greedy_repeat(num, k)
    print(f"{num!r}, k={k} -> {got!r}")
    assert got == expected

'1432219', k=3 -> '1219'
'10200', k=1 -> '200'
'10', k=2 -> '0'


In [9]:
# Code nộp LeetCode — Cách 2 (O(n*k), chỉ nên dùng khi n nhỏ)
class SolutionRepeat:
    def removeKdigits(self, num: str, k: int) -> str:
        s = num
        for _ in range(k):
            for i in range(len(s) - 1):
                if s[i] > s[i + 1]:
                    s = s[:i] + s[i + 1:]
                    break
            else:
                s = s[:-1]
        return s.lstrip("0") or "0"


print([SolutionRepeat().removeKdigits(num, k) for num, k, _ in examples])

['1219', '200', '0']


**Độ phức tạp Cách 2:** mỗi lần xóa quét `O(n)` và dựng chuỗi mới `O(n)` ⇒ Time `O(n * k)`; với `n = k = 10^5` là cỡ `10^10` — quá chậm. Space `O(n)`.

### Cách 3 — Monotonic stack (chốt)

**Giải thích từng đoạn:**

1. `stk = []` giữ các chữ số đang được giữ lại, luôn **không giảm** từ đáy lên đỉnh (nếu còn lượt xóa).
2. Duyệt từng chữ số `c`. Khi `k > 0`, stack không rỗng và `stk[-1] > c`: pop đỉnh (xóa chữ số lớn đứng trước chữ số nhỏ hơn), `k -= 1`.
3. Sau đó `stk.append(c)`.
4. Hết vòng lặp, nếu `k` còn dư (dãy không giảm) thì xóa `k` chữ số **cuối**: lấy `stk[:len(stk) - k]` (dùng `len(stk) - k` thay cho `-k` để an toàn khi `k = 0`).
5. Ghép chuỗi, `lstrip('0')`, `or '0'`.

In [10]:
def remove_k_stack(num, k):
    stk = []
    for c in num:
        while k > 0 and stk and stk[-1] > c:       # đỉnh lớn hơn chữ số mới -> xóa đỉnh
            stk.pop()
            k -= 1
        stk.append(c)
    if k > 0:                                      # còn lượt xóa: xóa các chữ số cuối
        stk = stk[:len(stk) - k]
    return "".join(stk).lstrip("0") or "0"


for num, k, expected in examples:
    got = remove_k_stack(num, k)
    print(f"{num!r}, k={k} -> {got!r}")
    assert got == expected

'1432219', k=3 -> '1219'
'10200', k=1 -> '200'
'10', k=2 -> '0'


In [11]:
# Code nộp LeetCode — Cách 3 (monotonic stack, O(n) time, O(n) space)
class SolutionStack:
    def removeKdigits(self, num: str, k: int) -> str:
        stk = []
        for c in num:
            while k > 0 and stk and stk[-1] > c:
                stk.pop()
                k -= 1
            stk.append(c)
        if k > 0:
            stk = stk[:len(stk) - k]
        return "".join(stk).lstrip("0") or "0"


print([SolutionStack().removeKdigits(num, k) for num, k, _ in examples])

['1219', '200', '0']


**Độ phức tạp Cách 3:** mỗi chữ số được `append` đúng một lần và `pop` tối đa một lần ⇒ tổng số thao tác `O(n)` (phân tích **amortized**: vòng `while` lồng bên trong không làm tăng bậc). Time `O(n)`, Space `O(n)`.

### Cách 4 — Chọn từng chữ số của kết quả (cửa sổ + danh sách vị trí)

**Ý tưởng khác hẳn:** kết quả có `m = n - k` chữ số. Ta chọn chúng **từng chữ số một**, từ trái sang phải. Giả sử đã chọn xong chữ số ở vị trí `prev` trong `num` (ban đầu `prev = -1`). Chữ số thứ `j` (đếm từ `0`) phải nằm ở một vị trí `p` thỏa `prev < p <= n - (m - j)`: để sau `p` vẫn còn đủ `m - j - 1` chữ số cho các lần chọn kế tiếp. **Trong cửa sổ đó, chọn chữ số nhỏ nhất, và nếu hòa thì chọn vị trí sớm nhất** (để chừa nhiều lựa chọn hơn cho sau).

**Cài đặt nhanh:** với mỗi chữ số `d` từ `0` đến `9` giữ danh sách các vị trí của nó (đã tăng dần). Để tìm vị trí đầu tiên `>= prev + 1` của `d`, dùng `bisect_left`, rồi kiểm tra có `<=` cận phải của cửa sổ không. Thử `d` từ nhỏ đến lớn, gặp cái đầu tiên hợp lệ là dừng. Mỗi lần chọn tốn tối đa `10 * log n`.

In [12]:
def remove_k_pick(num, k):
    n = len(num)
    m = n - k
    positions = [[] for _ in range(10)]             # positions[d] = các vị trí của chữ số d (tăng dần)
    for i, ch in enumerate(num):
        positions[int(ch)].append(i)
    res = []
    prev = -1
    for j in range(m):
        limit = n - (m - j)                         # vị trí lớn nhất còn cho phép
        for d in range(10):
            lst = positions[d]
            idx = bisect_left(lst, prev + 1)        # vị trí đầu tiên >= prev + 1 của chữ số d
            if idx < len(lst) and lst[idx] <= limit:
                res.append(str(d))
                prev = lst[idx]
                break
    return "".join(res).lstrip("0") or "0"


for num, k, expected in examples:
    got = remove_k_pick(num, k)
    print(f"{num!r}, k={k} -> {got!r}")
    assert got == expected

'1432219', k=3 -> '1219'
'10200', k=1 -> '200'
'10', k=2 -> '0'


In [13]:
# Code nộp LeetCode — Cách 4 (O(10 * n * log n) time, O(n) space)
from bisect import bisect_left


class SolutionPick:
    def removeKdigits(self, num: str, k: int) -> str:
        n = len(num)
        m = n - k
        positions = [[] for _ in range(10)]
        for i, ch in enumerate(num):
            positions[ord(ch) - 48].append(i)
        res = []
        prev = -1
        for j in range(m):
            limit = n - (m - j)
            for d in range(10):
                lst = positions[d]
                idx = bisect_left(lst, prev + 1)
                if idx < len(lst) and lst[idx] <= limit:
                    res.append(chr(48 + d))
                    prev = lst[idx]
                    break
        return "".join(res).lstrip("0") or "0"


print([SolutionPick().removeKdigits(num, k) for num, k, _ in examples])

['1219', '200', '0']


**Độ phức tạp Cách 4:** `m` lần chọn, mỗi lần thử tối đa 10 chữ số với một `bisect` `O(log n)` ⇒ Time `O(10 * m * log n)`, Space `O(n)` cho các danh sách vị trí. Cách này chậm hơn stack nhưng cho một góc nhìn khác: bài toán thực chất là **"tìm dãy con nhỏ nhất theo từ điển có độ dài `n - k`"**.

## 8. TRACE từng bước

Trace Cách 3 với `num = "1432219"`, `k = 3`:

| bước | chữ số `c` | stack trước | thao tác pop | `k` sau pop | stack sau `append` |
|---|---|---|---|---|---|
| 1 | `1` | `[]` | không | 3 | `[1]` |
| 2 | `4` | `[1]` | `1 > 4`? không | 3 | `[1,4]` |
| 3 | `3` | `[1,4]` | pop `4` | 2 | `[1,3]` |
| 4 | `2` | `[1,3]` | pop `3` | 1 | `[1,2]` |
| 5 | `2` | `[1,2]` | `2 > 2`? không | 1 | `[1,2,2]` |
| 6 | `1` | `[1,2,2]` | pop `2` | 0 | `[1,2,1]` |
| 7 | `9` | `[1,2,1]` | `k = 0` nên dừng | 0 | `[1,2,1,9]` |

Kết quả `"1219"` ✓. Hàm `trace_stack` in đúng các bước này; `trace_pick` in cửa sổ chọn của Cách 4.

In [14]:
def trace_stack(num, k):
    print(f"num={num!r}, k={k}")
    stk = []
    for step, c in enumerate(num, 1):
        before = "".join(stk)
        popped = []
        while k > 0 and stk and stk[-1] > c:
            popped.append(stk.pop())
            k -= 1
        stk.append(c)
        print(f"bước {step}: c={c} | stack trước={before!r:>9} | pop={popped} | k={k} | stack sau={''.join(stk)!r}")
    if k > 0:
        print(f"còn k={k} lượt xóa: cắt {k} chữ số cuối của {''.join(stk)!r}")
        stk = stk[:len(stk) - k]
    raw = "".join(stk)
    res = raw.lstrip("0") or "0"
    print(f"ghép = {raw!r} -> bỏ số 0 đầu -> {res!r}")
    return res


trace_stack("1432219", 3)

num='1432219', k=3
bước 1: c=1 | stack trước=       '' | pop=[] | k=3 | stack sau='1'
bước 2: c=4 | stack trước=      '1' | pop=[] | k=3 | stack sau='14'
bước 3: c=3 | stack trước=     '14' | pop=['4'] | k=2 | stack sau='13'
bước 4: c=2 | stack trước=     '13' | pop=['3'] | k=1 | stack sau='12'
bước 5: c=2 | stack trước=     '12' | pop=[] | k=1 | stack sau='122'
bước 6: c=1 | stack trước=    '122' | pop=['2'] | k=0 | stack sau='121'
bước 7: c=9 | stack trước=    '121' | pop=[] | k=0 | stack sau='1219'
ghép = '1219' -> bỏ số 0 đầu -> '1219'


'1219'

In [15]:
trace_stack("10200", 1)
print()
trace_stack("12345", 2)       # dãy tăng: phải cắt ở cuối
print()
trace_stack("10", 2)          # xóa hết -> "0"

num='10200', k=1
bước 1: c=1 | stack trước=       '' | pop=[] | k=1 | stack sau='1'
bước 2: c=0 | stack trước=      '1' | pop=['1'] | k=0 | stack sau='0'
bước 3: c=2 | stack trước=      '0' | pop=[] | k=0 | stack sau='02'
bước 4: c=0 | stack trước=     '02' | pop=[] | k=0 | stack sau='020'
bước 5: c=0 | stack trước=    '020' | pop=[] | k=0 | stack sau='0200'
ghép = '0200' -> bỏ số 0 đầu -> '200'

num='12345', k=2
bước 1: c=1 | stack trước=       '' | pop=[] | k=2 | stack sau='1'
bước 2: c=2 | stack trước=      '1' | pop=[] | k=2 | stack sau='12'
bước 3: c=3 | stack trước=     '12' | pop=[] | k=2 | stack sau='123'
bước 4: c=4 | stack trước=    '123' | pop=[] | k=2 | stack sau='1234'
bước 5: c=5 | stack trước=   '1234' | pop=[] | k=2 | stack sau='12345'
còn k=2 lượt xóa: cắt 2 chữ số cuối của '12345'
ghép = '123' -> bỏ số 0 đầu -> '123'

num='10', k=2
bước 1: c=1 | stack trước=       '' | pop=[] | k=2 | stack sau='1'
bước 2: c=0 | stack trước=      '1' | pop=['1'] | k=1 | stack sau='0'
c

'0'

In [16]:
def trace_pick(num, k):
    n, m = len(num), len(num) - k
    positions = [[] for _ in range(10)]
    for i, ch in enumerate(num):
        positions[int(ch)].append(i)
    print(f"num={num!r}, k={k}, cần giữ m={m} chữ số")
    res, prev = [], -1
    for j in range(m):
        limit = n - (m - j)
        window = num[prev + 1: limit + 1]
        for d in range(10):
            lst = positions[d]
            idx = bisect_left(lst, prev + 1)
            if idx < len(lst) and lst[idx] <= limit:
                print(f"lần {j + 1}: cửa sổ vị trí [{prev + 1}..{limit}] = {window!r} -> chữ số nhỏ nhất {d} ở vị trí {lst[idx]}")
                res.append(str(d))
                prev = lst[idx]
                break
    out = "".join(res)
    print("ghép =", repr(out), "->", repr(out.lstrip("0") or "0"))


trace_pick("1432219", 3)

num='1432219', k=3, cần giữ m=4 chữ số
lần 1: cửa sổ vị trí [0..3] = '1432' -> chữ số nhỏ nhất 1 ở vị trí 0
lần 2: cửa sổ vị trí [1..4] = '4322' -> chữ số nhỏ nhất 2 ở vị trí 3
lần 3: cửa sổ vị trí [4..5] = '21' -> chữ số nhỏ nhất 1 ở vị trí 5
lần 4: cửa sổ vị trí [6..6] = '9' -> chữ số nhỏ nhất 9 ở vị trí 6
ghép = '1219' -> '1219'


## 9. Code nộp LeetCode

Lời giải chính của notebook là **Cách 3** (monotonic stack): ngắn gọn, `O(n)` thời gian.

In [17]:
class Solution:
    def removeKdigits(self, num: str, k: int) -> str:
        stk = []                                    # stack các chữ số được giữ lại (không giảm)
        for c in num:
            while k > 0 and stk and stk[-1] > c:    # đỉnh lớn hơn chữ số mới -> xóa đỉnh
                stk.pop()
                k -= 1
            stk.append(c)
        if k > 0:                                   # dãy không giảm: xóa k chữ số cuối
            stk = stk[:len(stk) - k]
        return "".join(stk).lstrip("0") or "0"     # bỏ 0 ở đầu; rỗng -> "0"


for num, k, expected in examples:
    got = Solution().removeKdigits(num, k)
    print(f"{num!r}, k={k} -> {got!r}")
    assert got == expected

'1432219', k=3 -> '1219'
'10200', k=1 -> '200'
'10', k=2 -> '0'


## 10. Vì sao đúng

1. **Bất biến của stack:** tại mọi thời điểm, nếu `k` (lượt xóa còn lại) chưa hết thì stack là dãy **không giảm** từ đáy lên đỉnh, vì ta pop mọi đỉnh lớn hơn chữ số mới trước khi push.
2. **Mỗi lần pop là hợp lý:** khi `stk[-1] > c`, chữ số `stk[-1]` đứng trước chữ số nhỏ hơn `c`. Theo lập luận ở mục 6, xóa nó làm số nhỏ đi ở vị trí của nó và không ảnh hưởng các vị trí bên trái. Mọi chữ số bên trái nó trong stack đều `<=` nó nên không có chữ số "đáng xóa hơn" ở bên trái.
3. **Còn lượt xóa sau vòng lặp:** nghĩa là chưa từng có cặp giảm nào mới (stack không giảm toàn bộ). Khi đó chữ số **cuối** là lớn nhất-và-ít quan trọng nhất; xóa `k` chữ số cuối là tối ưu.
4. **Độ dài đúng:** mỗi lần pop trừ `k` đúng 1 và đúng 1 chữ số bị loại, phần dư cắt đúng `k` ⇒ tổng xóa đúng `k`, kết quả có đúng `n - k` chữ số trước khi bỏ `0`.
5. **Bỏ `0` đầu không làm đổi giá trị:** so sánh các ứng viên cùng độ dài (có thể có `0` đầu) tương đương so sánh giá trị; bỏ `0` đầu chỉ là cách viết. Kết quả rỗng/toàn `0` ⇒ `"0"`.

## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn mọi tổ hợp | `O(C(n,k) * n)` | `O(n)` |
| 2. Xóa tham lam từng lần | `O(n * k)` | `O(n)` |
| 3. Monotonic stack | **`O(n)`** | **`O(n)`** |
| 4. Chọn từng chữ số + `bisect` | `O(10 * n * log n)` | `O(n)` |

Với `n = 10^5`, Cách 3 chỉ cần khoảng `10^5` thao tác push/pop.

## 12. Bộ test

Oracle độc lập: `remove_k_brute` (mục 5) thử **mọi cách** giữ lại `n - k` chữ số. Ta dùng nó cho mọi input nhỏ; với input lớn đối chiếu giữa Cách 3 và Cách 4 (hai ý tưởng khác nhau) và kiểm tra tính chất.

In [18]:
SOLUTIONS = [Solution(), SolutionRepeat(), SolutionStack(), SolutionPick()]

# 12.1 Ví dụ của đề + edge case
cases = [
    ("1432219", 3, "1219"),
    ("10200", 1, "200"),
    ("10", 2, "0"),
    ("9", 1, "0"),                  # xóa hết
    ("112", 1, "11"),               # chữ số bằng nhau: không pop khi bằng
    ("12345", 2, "123"),            # dãy tăng: cắt ở cuối
    ("54321", 2, "321"),            # dãy giảm: pop liên tục
    ("1000", 1, "0"),               # còn lại toàn 0
    ("100", 1, "0"),
    ("1234567890", 9, "0"),
    ("10001", 4, "0"),
    ("10001", 3, "0"),
    ("10001", 2, "0"),
    ("200", 1, "0"),
    ("1173", 2, "11"),
    ("5337", 2, "33"),
    ("9999", 2, "99"),
    ("0", 1, "0"),
]
for num, k, expected in cases:
    for sol in SOLUTIONS:
        got = sol.removeKdigits(num, k)
        assert got == expected, (type(sol).__name__, num, k, got, expected)
print("OK:", len(cases), "test x", len(SOLUTIONS), "lớp")

OK: 18 test x 4 lớp


In [19]:
# 12.2 Đối chiếu với oracle vét cạn: MỌI chuỗi chữ số độ dài <= 6 trên bảng chữ cái {0,1,2,9}, mọi k
from itertools import product

count = 0
for n in range(1, 7):
    for tup in product("0129", repeat=n):
        num = "".join(tup)
        if n > 1 and num[0] == "0":
            continue                                  # đề: không có 0 thừa ở đầu
        for k in range(1, n + 1):
            expected = remove_k_brute(num, k)
            for sol in SOLUTIONS:
                got = sol.removeKdigits(num, k)
                assert got == expected, (type(sol).__name__, num, k, got, expected)
            count += 1
print("OK: đối chiếu", count, "cặp (num, k) với vét cạn")

OK: đối chiếu 23212 cặp (num, k) với vét cạn


In [20]:
# 12.3 Ngẫu nhiên có seed: độ dài <= 12, chữ số 0..9, đối chiếu với vét cạn
random.seed(402)
for _ in range(1500):
    n = random.randint(1, 12)
    first = random.choice("123456789") if n > 1 else random.choice("0123456789")
    num = first + "".join(random.choice("0123456789") for _ in range(n - 1))
    k = random.randint(1, n)
    expected = remove_k_brute(num, k)
    for sol in SOLUTIONS:
        assert sol.removeKdigits(num, k) == expected, (type(sol).__name__, num, k)
print("OK: 1500 input ngẫu nhiên, độ dài <= 12")

OK: 1500 input ngẫu nhiên, độ dài <= 12


In [21]:
# 12.4 Input lớn (đúng ràng buộc): Cách 3 và Cách 4 phải thống nhất, kèm kiểm tra tính chất
random.seed(403)
for trial in range(6):
    n = random.choice([2000, 20000, 100000])
    alphabet = random.choice(["0123456789", "01", "09", "123"])
    num = random.choice("123456789") + "".join(random.choice(alphabet) for _ in range(n - 1))
    k = random.choice([1, n // 3, n // 2, n - 1, n])
    a = SolutionStack().removeKdigits(num, k)
    b = SolutionPick().removeKdigits(num, k)
    assert a == b, (n, k)
    assert a == "0" or a[0] != "0"                       # không có 0 thừa ở đầu
    # tính chất: kết quả (nếu không phải "0") là dãy con của num
    it = iter(num)
    assert a == "0" or all(ch in it for ch in a)
    assert len(a) <= n - k or a == "0"
    print(f"n={n:>6}, k={k:>6}, alphabet={alphabet:>10}: độ dài kết quả {len(a):>6}")
print("OK: Cách 3 và Cách 4 thống nhất trên input lớn")

n=  2000, k=   666, alphabet=        09: độ dài kết quả    639
n=  2000, k=   666, alphabet=        09: độ dài kết quả    710
n=  2000, k=  2000, alphabet=0123456789: độ dài kết quả      1
n=100000, k=     1, alphabet=       123: độ dài kết quả  99999
n= 20000, k= 10000, alphabet=       123: độ dài kết quả  10000


n=100000, k=     1, alphabet=0123456789: độ dài kết quả  99999
OK: Cách 3 và Cách 4 thống nhất trên input lớn


In [22]:
# 12.5 Hiệu năng: n = 10^5 với Cách 3 (chốt)
random.seed(404)
big = "9" + "".join(random.choice("0123456789") for _ in range(10**5 - 1))
t = time.perf_counter()
res = Solution().removeKdigits(big, 50_000)
dt = time.perf_counter() - t
assert dt < 2.0
print("n = 10^5, k = 50000 xong dưới 2 giây:", dt < 2.0, "| độ dài kết quả:", len(res))
# Cách 2 chậm hơn rất nhiều (O(n*k)) -> chỉ thử cỡ nhỏ để so sánh
small = big[:3000]
t = time.perf_counter(); r2 = SolutionRepeat().removeKdigits(small, 1500); d2 = time.perf_counter() - t
t = time.perf_counter(); r3 = SolutionStack().removeKdigits(small, 1500); d3 = time.perf_counter() - t
assert r2 == r3
print(f"n=3000, k=1500: Cách 2 mất {d2 * 1000:.1f} ms, Cách 3 mất {d3 * 1000:.1f} ms (cùng kết quả)")

n = 10^5, k = 50000 xong dưới 2 giây: True | độ dài kết quả: 44588
n=3000, k=1500: Cách 2 mất 3.2 ms, Cách 3 mất 0.2 ms (cùng kết quả)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: quên bỏ số `0` ở đầu

In [23]:
def wrong_no_strip(num, k):
    stk = []
    for c in num:
        while k > 0 and stk and stk[-1] > c:
            stk.pop()
            k -= 1
        stk.append(c)
    stk = stk[:len(stk) - k]
    return "".join(stk)                              # SAI: không lstrip('0'), không xử lý rỗng


print(repr(wrong_no_strip("10200", 1)), "| đúng:", repr(Solution().removeKdigits("10200", 1)))
print(repr(wrong_no_strip("10", 2)), "| đúng:", repr(Solution().removeKdigits("10", 2)))
assert wrong_no_strip("10200", 1) == "0200" != "200"
assert wrong_no_strip("10", 2) == ""

'0200' | đúng: '200'
'' | đúng: '0'


**Vì sao sai:** stack giữ nguyên chữ số `0` ở đầu (`"0200"`), và khi xóa hết thì chuỗi rỗng `""` — đề yêu cầu `"200"` và `"0"`. **Cách sửa:** `"".join(stk).lstrip("0") or "0"`.

### Lỗi 2: pop cả khi bằng nhau (`>=` thay vì `>`)

In [24]:
def wrong_ge(num, k):
    stk = []
    for c in num:
        while k > 0 and stk and stk[-1] >= c:        # SAI: dấu >= pop cả chữ số bằng nhau
            stk.pop()
            k -= 1
        stk.append(c)
    stk = stk[:len(stk) - k]
    return "".join(stk).lstrip("0") or "0"


print(wrong_ge("112", 1), "| đúng:", Solution().removeKdigits("112", 1))
bad = [(num, k) for num in ["112", "1123", "2233", "1221"] for k in (1, 2) if wrong_ge(num, k) != remove_k_brute(num, k)]
print("input sai:", bad)
assert wrong_ge("112", 1) == "12" and Solution().removeKdigits("112", 1) == "11"

12 | đúng: 11
input sai: [('112', 1), ('1123', 1), ('1123', 2), ('2233', 1), ('2233', 2)]


**Vì sao sai:** xóa một chữ số bằng chữ số ngay sau nó **không làm số nhỏ đi** (vị trí đó vẫn nhận cùng giá trị) nhưng tiêu tốn một lượt xóa; lượt xóa đó lẽ ra phải dành cho chữ số lớn hơn ở phía sau. Với `"112"`, `k = 1`: đáp án đúng xóa `2` ở cuối được `"11"`, bản sai xóa một chữ số `1` được `"12"`. **Cách sửa:** chỉ pop khi `stk[-1] > c` **nghiêm ngặt**.

### Lỗi 3: cắt `stk[:-k]` khi `k` đã về `0`

In [25]:
def wrong_slice(num, k):
    stk = []
    for c in num:
        while k > 0 and stk and stk[-1] > c:
            stk.pop()
            k -= 1
        stk.append(c)
    stk = stk[:-k]                                    # SAI: khi k == 0 thì stk[:-0] == []
    return "".join(stk).lstrip("0") or "0"


print(wrong_slice("1432219", 3), "| đúng:", Solution().removeKdigits("1432219", 3))
print(wrong_slice("12345", 2), "| đúng:", Solution().removeKdigits("12345", 2))
assert wrong_slice("1432219", 3) == "0" and Solution().removeKdigits("1432219", 3) == "1219"
assert wrong_slice("12345", 2) == "123"

0 | đúng: 1219
123 | đúng: 123


**Vì sao sai:** sau khi pop đủ `3` lần, `k = 0` và `stk[:-0]` là `stk[:0]` = **danh sách rỗng**. **Cách sửa:** dùng `stk[:len(stk) - k]` (hoặc `if k: stk = stk[:-k]`).

### Lỗi 4: đổi sang `int` với chuỗi dài

In [26]:
huge = "1" + "0" * 5000
try:
    int(huge)
    print("không lỗi")
except ValueError as e:
    print("ValueError:", str(e)[:70], "...")
print("giới hạn mặc định:", sys.get_int_max_str_digits(), "chữ số; đề cho tới 10^5 chữ số")
assert sys.get_int_max_str_digits() == 4300

ValueError: Exceeds the limit (4300 digits) for integer string conversion: value h ...
giới hạn mặc định: 4300 chữ số; đề cho tới 10^5 chữ số


**Vì sao sai:** cách "đổi sang số, so sánh số" cho `num` tới `10^5` chữ số sẽ gặp `ValueError` (và dù có nới giới hạn cũng rất chậm). **Cách sửa:** luôn làm việc trên **chuỗi** (hoặc list ký tự), chỉ so sánh từng ký tự.

### Lỗi 5: nối chuỗi trong vòng lặp lớn (`O(n²)`)

Cách 2 dựng lại chuỗi `s[:i] + s[i+1:]` mỗi lần xóa: với `n`, `k` lớn sẽ tốn `O(n * k)`. Mục 12.5 đã đo: cùng `n = 3000`, stack nhanh hơn rất nhiều. **Cách sửa:** dùng stack `list` và `"".join` đúng một lần ở cuối.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu nhận dạng:** "xóa/giữ `k` phần tử để dãy kết quả **nhỏ nhất/lớn nhất theo từ điển**" ⇒ **monotonic stack**. Cùng họ: *Remove Duplicate Letters* (316), *Smallest Subsequence of Distinct Characters* (1081), *Find the Most Competitive Subsequence* (1673), *Create Maximum Number* (321), *Next Greater Element* (496, 739).
- **Quy tắc viết nhanh:** muốn số **nhỏ nhất** → duy trì stack **tăng** (pop khi đỉnh `>` phần tử mới). Muốn **lớn nhất** → stack **giảm** (pop khi đỉnh `<` phần tử mới).
- **Pop khi bằng nhau không bao giờ có lợi** (không làm đổi giá trị mà mất lượt xóa).
- Sau vòng lặp **luôn xử lý `k` còn dư** và **bỏ `0` ở đầu**, **rỗng ⇒ `"0"`**.
- Với `k` lớn bằng `n` ⇒ đáp án `"0"`. Với `k = 0` (nếu có) ⇒ trả lại `num`.
- Kiểm chứng bằng vét cạn trên chuỗi ngắn trước khi tin vào tham lam.

## 15. Biến thể

1. **Số lớn nhất sau khi xóa `k` chữ số:** đổi dấu so sánh (pop khi `stk[-1] < c`).
2. **Dãy con cạnh tranh nhất độ dài `k`** (LeetCode 1673): giữ đúng `k` phần tử nhỏ nhất theo từ điển, tương đương xóa `n - k` phần tử.
3. **Xóa các ký tự trùng lặp để chuỗi nhỏ nhất theo từ điển** (316, 1081): thêm điều kiện "chỉ pop nếu ký tự đó còn xuất hiện về sau".
4. **Xóa `k` chữ số sao cho số *chia hết* cho một số nào đó:** cần DP, không còn tham lam thuần túy.

In [27]:
def remove_k_largest(num, k):
    """Số LỚN NHẤT sau khi xóa đúng k chữ số (giữ lại độ dài n - k)."""
    stk = []
    for c in num:
        while k > 0 and stk and stk[-1] < c:
            stk.pop()
            k -= 1
        stk.append(c)
    return "".join(stk[:len(stk) - k])


def remove_k_largest_brute(num, k):
    m = len(num) - k
    return max("".join(num[i] for i in keep) for keep in combinations(range(len(num)), m))


assert remove_k_largest("1432219", 3) == "4329"
random.seed(4021)
for _ in range(500):
    n = random.randint(2, 9)
    num = "".join(random.choice("0123456789") for _ in range(n))
    k = random.randint(1, n - 1)
    assert remove_k_largest(num, k) == remove_k_largest_brute(num, k), (num, k)
print("số lớn nhất của '1432219' khi xóa 3 chữ số:", remove_k_largest("1432219", 3))


def most_competitive(nums, k):
    """LeetCode 1673: dãy con độ dài k nhỏ nhất theo từ điển."""
    stk = []
    n = len(nums)
    for i, x in enumerate(nums):
        while stk and stk[-1] > x and len(stk) - 1 + (n - i) >= k:
            stk.pop()
        if len(stk) < k:
            stk.append(x)
    return stk


print(most_competitive([3, 5, 2, 6], 2), most_competitive([2, 4, 3, 3, 5, 4, 9, 6], 4))
assert most_competitive([3, 5, 2, 6], 2) == [2, 6]
assert most_competitive([2, 4, 3, 3, 5, 4, 9, 6], 4) == [2, 3, 3, 4]

số lớn nhất của '1432219' khi xóa 3 chữ số: 4329
[2, 6] [2, 3, 3, 4]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn tổ hợp | thử mọi cách giữ `n - k` chữ số | `O(C(n,k) * n)` | `O(n)` | **oracle** kiểm chứng trên input nhỏ |
| 2. Xóa tham lam từng lần | xóa chữ số đầu tiên lớn hơn chữ số kế tiếp, lặp `k` lần | `O(n * k)` | `O(n)` | hiểu quy tắc tham lam, `n` nhỏ |
| 3. Monotonic stack | một lượt, pop khi đỉnh > chữ số mới | **`O(n)`** | `O(n)` | **nộp bài**, phỏng vấn |
| 4. Chọn từng chữ số | chọn chữ số nhỏ nhất trong cửa sổ hợp lệ | `O(10 * n * log n)` | `O(n)` | góc nhìn "dãy con nhỏ nhất theo từ điển" |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `"0200".lstrip("0")`; (b) `"000".lstrip("0") or "0"`; (c) `"abc"[:-0]`; (d) kết quả của `Solution().removeKdigits("9", 1)`.

**Bài 2 (tính tay).** Bằng cách trace stack, tìm kết quả của `num = "30200"`, `k = 1` và `num = "9112"`, `k = 2`. Ghi lại stack sau mỗi chữ số.

**Bài 3 (điền TODO).** Hoàn thành `remove_k_todo(num, k)` theo Cách 3.

In [28]:
def remove_k_todo(num, k):
    # TODO 1: stk = []
    # TODO 2: với mỗi c trong num: while k > 0 và stk và stk[-1] > c: pop và giảm k; rồi append c
    # TODO 3: nếu k còn dư, cắt k chữ số cuối: stk = stk[:len(stk) - k]
    # TODO 4: trả về "".join(stk).lstrip("0") or "0"
    return None

# KHÔNG gọi hàm TODO ở đây để Run All không lỗi (TODO chưa điền).

**Bài 4 (tự viết lại).** Viết `remove_k_pick2(num, k)` theo Cách 4 (chọn từng chữ số bằng cửa sổ) **không nhìn lại** mục 7. Kiểm tra với `[("1432219", 3), ("10200", 1), ("10", 2)]`.

**Bài 5 (biến thể / mini challenge).** Viết `remove_k_nonadjacent_equal(num, k)`: giống bài gốc nhưng trả về **số LỚN NHẤT** có thể sau khi xóa đúng `k` chữ số (không bỏ `0` đầu vì số lớn nhất không bắt đầu bằng `0` trừ khi cả dãy là `0`). Kiểm tra bằng vét cạn trên chuỗi ngắn (mục 15 đã có gợi ý).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `"200"`, `"0"`, `""` (vì `-0 == 0`), `"0"`. Bài 3: xem hàm `remove_k_stack` ở mục 7.
- **Mức 2 (vừa):** Bài 2: `"30200"`, `k = 1`: `3` vào; `0 < 3` pop `3` (k = 0), push `0`; rồi `2, 0, 0` ⇒ `"0200"` ⇒ `"200"`. `"9112"`, `k = 2`: `9`; `1 < 9` pop (k = 1), push `1`; `1` (bằng, không pop); `2` ⇒ stack `1,1,2`, còn `k = 1` ⇒ cắt cuối ⇒ `"11"`. Bài 4: `m = n - k`, `limit = n - (m - j)`.
- **Mức 3 (sâu):** Bài 5: đổi điều kiện pop thành `stk[-1] < c`; còn `k` dư vẫn cắt ở cuối; so sánh với `max(...)` trên các tổ hợp bằng `combinations`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [29]:
# Bài 1
assert "0200".lstrip("0") == "200"
assert ("000".lstrip("0") or "0") == "0"
assert "abc"[:-0] == ""
assert Solution().removeKdigits("9", 1) == "0"

# Bài 2
assert Solution().removeKdigits("30200", 1) == "200"
assert Solution().removeKdigits("9112", 2) == "11"


# Bài 3
def remove_k_todo_ans(num, k):
    stk = []
    for c in num:
        while k > 0 and stk and stk[-1] > c:
            stk.pop()
            k -= 1
        stk.append(c)
    if k > 0:
        stk = stk[:len(stk) - k]
    return "".join(stk).lstrip("0") or "0"


for num, k, expected in examples:
    assert remove_k_todo_ans(num, k) == expected


# Bài 4
def remove_k_pick2(num, k):
    n, m = len(num), len(num) - k
    pos = {d: [] for d in "0123456789"}
    for i, ch in enumerate(num):
        pos[ch].append(i)
    out, prev = [], -1
    for j in range(m):
        limit = n - (m - j)
        for d in "0123456789":
            idx = bisect_left(pos[d], prev + 1)
            if idx < len(pos[d]) and pos[d][idx] <= limit:
                out.append(d)
                prev = pos[d][idx]
                break
    return "".join(out).lstrip("0") or "0"


for num, k, expected in examples:
    assert remove_k_pick2(num, k) == expected
random.seed(4022)
for _ in range(300):
    n = random.randint(1, 10)
    num = random.choice("123456789") + "".join(random.choice("0123456789") for _ in range(n - 1))
    k = random.randint(1, n)
    assert remove_k_pick2(num, k) == remove_k_brute(num, k)


# Bài 5
def remove_k_nonadjacent_equal(num, k):
    stk = []
    for c in num:
        while k > 0 and stk and stk[-1] < c:
            stk.pop()
            k -= 1
        stk.append(c)
    return "".join(stk[:len(stk) - k])


assert remove_k_nonadjacent_equal("1432219", 3) == "4329"
assert remove_k_nonadjacent_equal("10", 1) == "1"
for _ in range(300):
    n = random.randint(2, 9)
    num = "".join(random.choice("0123456789") for _ in range(n))
    k = random.randint(1, n - 1)
    assert remove_k_nonadjacent_equal(num, k) == remove_k_largest_brute(num, k)
print("OK: toàn bộ đáp án đúng")

OK: toàn bộ đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template** (monotonic stack tạo số nhỏ nhất)

```python
class Solution:
    def removeKdigits(self, num: str, k: int) -> str:
        stk = []
        for c in num:
            while k > 0 and stk and stk[-1] > c:
                stk.pop()
                k -= 1
            stk.append(c)
        if k > 0:
            stk = stk[:len(stk) - k]
        return "".join(stk).lstrip("0") or "0"
```

**Checklist**

- [ ] Điều kiện pop dùng `>` (không phải `>=`) và có `k > 0 and stk`?
- [ ] Xử lý `k` còn dư sau vòng lặp bằng `stk[:len(stk) - k]` (an toàn khi `k = 0`)?
- [ ] Bỏ `0` ở đầu bằng `lstrip("0")` và trả về `"0"` khi rỗng?
- [ ] Không đổi sang `int` (chuỗi tới `10^5` chữ số)?
- [ ] Đã thử `"10"`, `k = 2`; `"10200"`, `k = 1`; `"112"`, `k = 1`; dãy tăng `"12345"`; dãy giảm `"54321"`?
- [ ] Đã đối chiếu với vét cạn trên chuỗi ngắn?